In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import*
from pyspark.sql.window import*
from pyspark.sql.types import*
spark=SparkSession.builder.appName("bike oriject").getOrCreate()
df=spark.read.csv("/Volumes/bike_catalog/bike_schema/bike_volume", header=True)
#df.show()
df.printSchema()
df_bronze=df.withColumn("started_at",col("started_at").cast(TimestampType()))\
            .withColumn("ended_at",col("ended_at").cast(TimestampType()))\
           .withColumn("start_lat",col("start_lat").cast(DecimalType(10,6)))\
               .withColumn("start_lng",col("start_lng").cast(DecimalType(10,6)))\
                   .withColumn("end_lat",col("end_lat").cast(DecimalType(10,6)))\
                       .withColumn("end_lng",col("end_lng").cast(DecimalType(10,6)))              
#df.printSchema()
df_bronze.printSchema()
#df_bronze.show()
df_silver=df_bronze.select("ride_id",col("started_at").cast(DateType()).alias("trip_start_date"),
                           "started_at","ended_at","start_station_name","end_station_name",((unix_timestamp(col("ended_at"))-unix_timestamp(col("started_at")))/60).alias("trip_duration_mins").cast(DecimalType()))
#df_silver.show(100,60)
df_silver.printSchema()
df_gold1=df_silver.groupBy("trip_start_date").agg(max("trip_duration_mins").alias("max_trip_duration_mins"),min("trip_duration_mins").alias("min_trip_dutarion_mins"),max("trip_duration_mins").alias("avg_trip_duration_mins"),count("trip_duration_mins").alias("total_no_trips").cast(IntegerType()))
#df_gold1.show()
df_gold1.printSchema()

df_gold2=df_silver.groupby("trip_start_date","start_station_name").agg(avg("trip_duration_mins").alias("avg_trip_duration_mins"),count("trip_duration_mins").alias("total_no_trips").cast(IntegerType()))
#df_gold2.show()
df_gold2.printSchema()